## 非小细胞肺癌数据分析

### 定义一些绘图函数

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

import matplotlib
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import gridspec
from matplotlib.colors import LinearSegmentedColormap
# from plottable import ColumnDefinition, Table
# from plottable.cmap import normed_cmap
# from plottable.formatters import tickcross
# from plottable.plots import bar


default_color_dict = {
    "0": "#66C5CC",
    "1": "#F6CF71",
    "2": "#F89C74",
    "3": "#DCB0F2",
    "4": "#87C55F",
    "5": "#9EB9F3",
    "6": "#FE88B1",
    "7": "#C9DB74",
    "8": "#8BE0A4",
    "9": "#B497E7",
    "10": "#D3B484",
    "11": "#B3B3B3",
    "12": "#276A8C", # Royal Blue
    "13": "#DAB6C4", # Pink
    "14": "#C38D9E", # Mauve-Pink
    "15": "#9D88A2", # Mauve
    "16": "#FF4D4D", # Light Red
    "17": "#9B4DCA", # Lavender-Purple
    "18": "#FF9CDA", # Bright Pink
    "19": "#FF69B4", # Hot Pink
    "20": "#FF00FF", # Magenta
    "21": "#DA70D6", # Orchid
    "22": "#BA55D3", # Medium Orchid
    "23": "#8A2BE2", # Blue Violet
    "24": "#9370DB", # Medium Purple
    "25": "#7B68EE", # Medium Slate Blue
    "26": "#4169E1", # Royal Blue
    "27": "#FF8C8C", # Salmon Pink
    "28": "#FFAA80", # Light Coral
    "29": "#48D1CC", # Medium Turquoise
    "30": "#40E0D0", # Turquoise
    "31": "#00FF00", # Lime
    "32": "#7FFF00", # Chartreuse
    "33": "#ADFF2F", # Green Yellow
    "34": "#32CD32", # Lime Green
    "35": "#228B22", # Forest Green
    "36": "#FFD8B8", # Peach
    "37": "#008080", # Teal
    "38": "#20B2AA", # Light Sea Green
    "39": "#00FFFF", # Cyan
    "40": "#00BFFF", # Deep Sky Blue
    "41": "#4169E1", # Royal Blue
    "42": "#0000CD", # Medium Blue
    "43": "#00008B", # Dark Blue
    "44": "#8B008B", # Dark Magenta
    "45": "#FF1493", # Deep Pink
    "46": "#FF4500", # Orange Red
    "47": "#006400", # Dark Green
    "48": "#FF6347", # Tomato
    "49": "#FF7F50", # Coral
    "50": "#CD5C5C", # Indian Red
    "51": "#B22222", # Fire Brick
    "52": "#FFB83F",  # Light Orange
    "53": "#8B0000", # Dark Red
    "54": "#D2691E", # Chocolate
    "55": "#A0522D", # Sienna
    "56": "#800000", # Maroon
    "57": "#808080", # Gray
    "58": "#A9A9A9", # Dark Gray
    "59": "#C0C0C0", # Silver
    "60": "#9DD84A",
    "61": "#F5F5F5", # White Smoke
    "62": "#F17171", # Light Red
    "63": "#000000", # Black
    "64": "#FF8C42", # Tangerine
    "65": "#F9A11F", # Bright Orange-Yellow
    "66": "#FACC15", # Golden Yellow
    "67": "#E2E062", # Pale Lime
    "68": "#BADE92", # Soft Lime
    "69": "#70C1B3", # Greenish-Blue
    "70": "#41B3A3", # Turquoise
    "71": "#5EAAA8", # Gray-Green
    "72": "#72B01D", # Chartreuse
    "73": "#9CD08F", # Light Green
    "74": "#8EBA43", # Olive Green
    "75": "#FAC8C3", # Light Pink
    "76": "#E27D60", # Dark Salmon
    "77": "#C38D9E", # Mauve-Pink
    "78": "#937D64", # Light Brown
    "79": "#B1C1CC", # Light Blue-Gray
    "80": "#88A0A8", # Gray-Blue-Green
    "81": "#4E598C", # Dark Blue-Purple
    "82": "#4B4E6D", # Dark Gray-Blue
    "83": "#8E9AAF", # Light Blue-Grey
    "84": "#C0D6DF", # Pale Blue-Grey
    "85": "#97C1A9", # Blue-Green
    "86": "#4C6E5D", # Dark Green
    "87": "#95B9C7", # Pale Blue-Green
    "88": "#C1D5E0", # Pale Gray-Blue
    "89": "#ECDB54", # Bright Yellow
    "90": "#E89B3B", # Bright Orange
    "91": "#CE5A57", # Deep Red
    "92": "#C3525A", # Dark Red
    "93": "#B85D8E", # Berry
    "94": "#7D5295", # Deep Purple
    "-1" : "#E1D9D1",
    "None" : "#E1D9D1"
}


def create_new_color_dict(
        adata,
        cat_key,
        color_palette="default",
        overwrite_color_dict={"-1" : "#E1D9D1"},
        skip_default_colors=0):
    """
    Create a dictionary of color hexcodes for a specified category.

    Parameters
    ----------
    adata:
        AnnData object.
    cat_key:
        Key in ´adata.obs´ where the categories are stored for which color
        hexcodes will be created.
    color_palette:
        Type of color palette.
    overwrite_color_dict:
        Dictionary with overwrite values that will take precedence over the
        automatically created dictionary.
    skip_default_colors:
        Number of colors to skip from the default color dict.

    Returns
    ----------
    new_color_dict:
        The color dictionary with a hexcode for each category.
    """
    new_categories = adata.obs[cat_key].unique().tolist()
    if color_palette == "cell_type_30":
        # https://github.com/scverse/scanpy/blob/master/scanpy/plotting/palettes.py#L40
        new_color_dict = {key: value for key, value in zip(
            new_categories,
            ["#023fa5",
             "#7d87b9",
             "#bec1d4",
             "#d6bcc0",
             "#bb7784",
             "#8e063b",
             "#4a6fe3",
             "#8595e1",
             "#b5bbe3",
             "#e6afb9",
             "#e07b91",
             "#d33f6a",
             "#11c638",
             "#8dd593",
             "#c6dec7",
             "#ead3c6",
             "#f0b98d",
             "#ef9708",
             "#0fcfc0",
             "#9cded6",
             "#d5eae7",
             "#f3e1eb",
             "#f6c4e1",
             "#f79cd4",
             '#7f7f7f',
             "#c7c7c7",
             "#1CE6FF",
             "#336600"])}
    elif color_palette == "cell_type_20":
        # https://github.com/vega/vega/wiki/Scales#scale-range-literals (some adjusted)
        new_color_dict = {key: value for key, value in zip(
            new_categories,
            ['#1f77b4',
             '#ff7f0e',
             '#279e68',
             '#d62728',
             '#aa40fc',
             '#8c564b',
             '#e377c2',
             '#b5bd61',
             '#17becf',
             '#aec7e8',
             '#ffbb78',
             '#98df8a',
             '#ff9896',
             '#c5b0d5',
             '#c49c94',
             '#f7b6d2',
             '#dbdb8d',
             '#9edae5',
             '#ad494a',
             '#8c6d31'])}
    elif color_palette == "cell_type_10":
        # scanpy vega10
        new_color_dict = {key: value for key, value in zip(
            new_categories,
            ['#7f7f7f',
             '#ff7f0e',
             '#279e68',
             '#e377c2',
             '#17becf',
             '#8c564b',
             '#d62728',
             '#1f77b4',
             '#b5bd61',
             '#aa40fc'])}
    elif color_palette == "batch":
        # sns.color_palette("colorblind").as_hex()
        new_color_dict = {key: value for key, value in zip(
            new_categories,
            ['#0173b2', '#d55e00', '#ece133', '#ca9161', '#fbafe4',
             '#949494', '#de8f05', '#029e73', '#cc78bc', '#56b4e9',
             '#F0F8FF', '#FAEBD7', '#00FFFF', '#7FFFD4', '#F0FFFF',
             '#F5F5DC', '#FFE4C4', '#000000', '#FFEBCD', '#0000FF',
             '#8A2BE2', '#A52A2A', '#DEB887', '#5F9EA0', '#7FFF00',
             '#D2691E', '#FF7F50', '#6495ED', '#FFF8DC', '#DC143C'])}
    elif color_palette == "default":
        new_color_dict = {key: value for key, value in zip(new_categories, list(default_color_dict.values())[skip_default_colors:])}
    for key, val in overwrite_color_dict.items():
        new_color_dict[key] = val
    return new_color_dict

In [ ]:
import collections
import colorsys

def get_distinct_colors(n):
    """
    https://www.quora.com/How-do-I-generate-n-visually-distinct-RGB-colours-in-Python/answer/Karthik-Kumar-Viswanathan
    """
    hue_partition = 1 / (n + 1)
    colors = [
        colorsys.hsv_to_rgb(hue_partition * value, 1.0, 1.0) for value in range(0, n)
    ]
    return colors[::2] + colors[1::2]


def text_width(fig, ax, text, fontsize):
    text = ax.text(-100, 0, text, fontsize=fontsize)
    text_bb = text.get_window_extent(renderer=fig.canvas.get_renderer())
    text_bb = text_bb.transformed(fig.dpi_scale_trans.inverted())
    width = text_bb.width
    text.remove()
    return width


class Sankey:
    def __init__(
        self,
        x,
        y,
        colorside,
        plot_width=8,
        plot_height=8,
        gap=0.12,
        alpha=0.3,
        fontsize="small",
        left_order=None,
        mapping=None,
        colors=None,
        #                  colorside=None,
        tag=None,
        title=None,
        title_left=None,
        title_right=None,
        ax=None,
    ):
        self.X = x
        self.Y = y
        if ax:
            self.plot_width = ax.get_position().width * ax.figure.get_size_inches()[0]
            self.plot_height = ax.get_position().height * ax.figure.get_size_inches()[1]
        else:
            self.plot_width = plot_width
            self.plot_height = plot_height
        self.gap = gap
        self.alpha = alpha
        self.colors = colors
        self.colorside = colorside
        self.fontsize = fontsize
        self.tag = tag
        self.map = mapping is not None
        self.mapping = mapping
        self.mapping_colors = {
            "increase": "#1f721c",
            "decrease": "#ddc90f",
            "mistake": "#dd1616",
            "correct": "#dddddd",
            "novel": "#59a8d6",
        }
        self.title = title
        self.title_left = title_left
        self.title_right = title_right

        self.need_title = any(
            map(lambda x: x is not None, (title, title_left, title_right))
        )
        if self.need_title:
            self.plot_height -= 0.5

        self.init_figure(ax)

        self.flows = collections.Counter(zip(x, y))
        self.init_nodes(left_order)

        self.init_widths()
        # inches per 1 item in x and y
        self.resolution = (plot_height - gap * (len(self.left_nodes) - 1)) / len(x)
        if self.colors == None:
            if colorside == "left":
                self.colors = {
                    name: colour
                    for name, colour in zip(
                        self.left_nodes.keys(),
                        get_distinct_colors(len(self.left_nodes)),
                    )
                }
            elif colorside == "right":
                self.colors = {
                    name: colour
                    for name, colour in zip(
                        self.right_nodes.keys(),
                        get_distinct_colors(len(self.right_nodes)),
                    )
                }
            else:
                raise ValueError(
                    "colorside argument should be set either to 'left' or 'right'. Exiting."
                )

        self.init_offsets()

    def init_figure(self, ax):
        if ax is None:
            self.fig = plt.figure()
            self.ax = plt.Axes(self.fig, [0, 0, 1, 1])
            self.fig.add_axes(self.ax)
        self.fig = ax.figure
        self.ax = ax

    def init_nodes(self, left_order):
        left_nodes = {}
        right_nodes = {}
        left_offset = 0
        for (left, right), flow in self.flows.items():
            if left in left_nodes:
                left_nodes[left] += flow
            else:
                left_nodes[left] = flow
            if right in right_nodes:
                node = right_nodes[right]
                node[0] += flow
                if flow > node[2]:
                    node[1] = left
                    node[2] = flow
            else:
                right_nodes[right] = [flow, left, flow]

        self.left_nodes = collections.OrderedDict()
        self.left_nodes_idx = {}
        if left_order is None:
            key = lambda pair: -pair[1]
        else:
            left_order = list(left_order)
            key = lambda pair: left_order.index(pair[0])

        for name, flow in sorted(left_nodes.items(), key=key):
            self.left_nodes[name] = flow
            self.left_nodes_idx[name] = len(self.left_nodes_idx)

        left_names = list(self.left_nodes.keys())
        self.right_nodes = collections.OrderedDict()
        self.right_nodes_idx = {}
        for name, node in sorted(
            right_nodes.items(),
            key=lambda pair: (left_names.index(pair[1][1]), -pair[1][2]),
        ):
            self.right_nodes[name] = node[0]
            self.right_nodes_idx[name] = len(self.right_nodes_idx)

    def init_widths(self):
        self.left_width = max(
            (
                text_width(self.fig, self.ax, node, self.fontsize)
                for node in self.left_nodes
            )
        )
        if self.title_left:
            self.left_width = max(
                self.left_width,
                text_width(self.fig, self.ax, self.title_left, self.fontsize) / 2,
            )
        self.right_width = max(
            (
                text_width(self.fig, self.ax, node, self.fontsize)
                for node in self.right_nodes
            )
        )
        if self.title_right:
            self.right_width = max(
                self.right_width,
                text_width(self.fig, self.ax, self.title_right, self.fontsize) / 2,
            )

        self.right_stop = self.plot_width - self.left_width - self.right_width
        self.middle1_stop = self.right_stop * 9 / 20
        self.middle2_stop = self.right_stop * 11 / 20

    def init_offsets(self):
        self.offsets_l = {}
        self.offsets_r = {}

        offset = 0
        for name, flow in self.left_nodes.items():
            self.offsets_l[name] = offset
            offset += flow * self.resolution + self.gap

        offset = 0
        for name, flow in self.right_nodes.items():
            self.offsets_r[name] = offset
            offset += flow * self.resolution + self.gap

    def draw_flow(self, left, right, flow, node_offsets_l, node_offsets_r, colorside):
        P = matplotlib.path.Path

        flow *= self.resolution
        left_y = self.offsets_l[left] + node_offsets_l[left]
        right_y = self.offsets_r[right] + node_offsets_r[right]
        if self.need_title:
            left_y += 0.5
            right_y += 0.5
        node_offsets_l[left] += flow
        node_offsets_r[right] += flow
        if colorside == "left":
            color = self.colors[left]
        elif colorside == "right":
            color = self.colors[right]
        if self.mapping is not None:
            color = self.mapping_colors[self.mapping.category(left, right)]

        path_data = [
            (P.MOVETO, (0, -left_y)),
            (P.LINETO, (0, -left_y - flow)),
            (P.CURVE4, (self.middle1_stop, -left_y - flow)),
            (P.CURVE4, (self.middle2_stop, -right_y - flow)),
            (P.CURVE4, (self.right_stop, -right_y - flow)),
            (P.LINETO, (self.right_stop, -right_y)),
            (P.CURVE4, (self.middle2_stop, -right_y)),
            (P.CURVE4, (self.middle1_stop, -left_y)),
            (P.CURVE4, (0, -left_y)),
            (P.CLOSEPOLY, (0, -left_y)),
        ]
        codes, verts = zip(*path_data)
        path = P(verts, codes)
        patch = matplotlib.patches.PathPatch(
            path,
            facecolor=color,
            alpha=0.9 if flow < 0.02 else self.alpha,
            edgecolor="none",
        )
        self.ax.add_patch(patch)

    def draw_label(self, label, is_left):
        nodes = self.left_nodes if is_left else self.right_nodes
        offsets = self.offsets_l if is_left else self.offsets_r
        y = offsets[label] + nodes[label] * self.resolution / 2
        if self.need_title:
            y += 0.5

        self.ax.text(
            -0.1 if is_left else self.right_stop + 0.1,
            -y,
            label,
            horizontalalignment="right" if is_left else "left",
            verticalalignment="center",
            fontsize=self.fontsize,
        )

    def draw_titles(self):
        if self.title:
            self.ax.text(
                self.right_stop / 2,
                -0.25,
                self.title,
                horizontalalignment="center",
                verticalalignment="center",
                fontsize=self.fontsize,
                fontweight="bold",
            )
        if self.title_left:
            self.ax.text(
                -0.1,
                -0.25,
                self.title_left,
                horizontalalignment="right",
                verticalalignment="center",
                fontsize=self.fontsize,
            )
        if self.title_right:
            self.ax.text(
                self.right_stop + 0.1,
                -0.25,
                self.title_right,
                horizontalalignment="left",
                verticalalignment="center",
                fontsize=self.fontsize,
            )

    def draw(self, colorside):
        node_offsets_l = collections.Counter()
        node_offsets_r = collections.Counter()

        for (left, right), flow in sorted(
            self.flows.items(),
            key=lambda pair: (
                self.left_nodes_idx[pair[0][0]],
                self.right_nodes_idx[pair[0][1]],
            ),
        ):
            self.draw_flow(left, right, flow, node_offsets_l, node_offsets_r, colorside)

        for name in self.left_nodes:
            self.draw_label(name, True)
        for name in self.right_nodes:
            self.draw_label(name, False)
        self.draw_titles()

        self.ax.axis("equal")
        self.ax.set_xlim(
            -self.left_width - self.gap, self.right_stop + self.gap + self.right_width
        )
        self.ax.get_xaxis().set_visible(False)
        self.ax.get_yaxis().set_visible(False)
        for k in self.ax.spines.keys():
            self.ax.spines[k].set_visible(False)
        # plt.axis('off')
        # self.fig.set_figheight(self.plot_height)
        # self.fig.set_figwidth(self.plot_width)
        if self.tag:
            text_ax = self.fig.add_axes((0.02, 0.95, 0.05, 0.05), frame_on=False)
            text_ax.set_axis_off()
            plt.text(
                0, 0, self.tag, fontsize=30, transform=text_ax.transAxes
            )
        # plt.tight_layout()


def sankey(x, y, colorside="left", **kwargs):
    diag = Sankey(x, y, colorside, **kwargs)
    diag.draw(colorside)
    return diag.fig

In [ ]:
import os
import pandas as pd
import numpy as np

os.chdir('/pri_exthome/zhouwg/project/Garfield')
os.getcwd()

In [ ]:
# load packages
import os
import warnings
import Garfield as gf
import scanpy as sc
import numpy as np
import pandas as pd
from mudata import MuData
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

gf.__version__

In [ ]:
Batch_list = []
section_ids = ['batch1', 'batch2', 'batch3', 'batch4',
               'batch5', 'batch6', 'batch7', 'batch8']
root_dir = '/pri_exthome/zhouwg/project/spatial_data/gold'
dataset_names = 'nanostring_cosmx_human_nsclc'

for section_id in section_ids:
    print('preprocessing:', section_id)
    input_dir = os.path.join(root_dir, f'{dataset_names}_{section_id}.h5ad')
    adata = sc.read_h5ad(input_dir)
    adata.X = adata.layers['counts'].copy()
    adata.var_names_make_unique()
    Batch_list.append(adata)

# Concat the scanpy objects for multiple slices
adata = Batch_list[0].concatenate(Batch_list[1:], batch_key='Batch')
adata

In [ ]:
adata.obs['batch'].value_counts()

In [ ]:
adata.obs['patient'].value_counts()

In [ ]:
adata.obs['cell_type_original'].value_counts()

In [ ]:
adata.obs['cell_type'].value_counts()

In [ ]:
# Ensure adata.X is counts.
# adata.layers['counts'] = adata.X.copy()
# adata.X = adata.layers['counts'].copy()
adata.X.max()

In [ ]:
### spatial reference building 选择除lung13之外的所有样本
adata_new = adata[~adata.obs['batch'].isin(['lung13'])].copy()
adata_new

In [ ]:
adata_new.obs['batch'].value_counts()

### Integrating spatially resolved transcriptomics data using Garfield

In [ ]:
# set workdir #
workdir = f'/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_NSCLC'
gf.settings.set_workdir(workdir)

### modify parameter
user_config = dict(
    ## Input options
    adata_list=adata_new,
    profile='spatial',
    data_type='single-modal',
    sample_col=None, # batch
    weight=0.5,
    ## Preprocessing options
    graph_const_method='Squidpy', # mu_std, Radius, KNN, Squidpy
    used_hvg=True,
    min_cells=3,
    min_features=0,
    keep_mt=False,
    target_sum=1e4,
    rna_n_top_features=3000,
    n_components=50,
    n_neighbors=5,
    metric='euclidean',
    svd_solver='arpack',
    # datasets
    used_pca_feat=False,
    adj_key='connectivities',
    # data split parameters
    edge_val_ratio=0.1,
    edge_test_ratio=0.,
    node_val_ratio=0.1,
    node_test_ratio=0.,
    ## Model options
    augment_type='svd', # svd
    svd_q=5,
    use_FCencoder=True,
    conv_type='GAT', # GAT or GATv2Conv or GCN
    gnn_layer=2,
    hidden_dims=[128, 128],
    bottle_neck_neurons=20,
    cluster_num=20,
    drop_feature_rate=0.2,
    drop_edge_rate=0.2,
    num_heads=3,
    dropout=0.2,
    concat=True,
    used_edge_weight=True,
    used_DSBN=False,
    used_mmd=True,
    # data loader parameters
    num_neighbors=5,
    loaders_n_hops=2,
    edge_batch_size=4096,
    node_batch_size=512, # None
    # loss parameters
    include_edge_recon_loss=True,
    include_gene_expr_recon_loss=True,
    lambda_latent_contrastive_instanceloss=1.0,
    lambda_latent_contrastive_clusterloss=0.5,
    lambda_gene_expr_recon=1., #
    lambda_edge_recon=10., #
    lambda_latent_adj_recon_loss=2.,
    lambda_omics_recon_mmd_loss=0.5,
    # train parameters
    n_epochs_no_edge_recon=0,
    learning_rate=0.001,
    weight_decay=1e-05,
    gradient_clipping=5,
    # other parameters
    latent_key='garfield_latent',
    reload_best_model=True,
    use_early_stopping=True,
    early_stopping_kwargs=None,
    monitor=True,
    device_id=0,
    seed=2024,
    verbose=True
)
dict_config = gf.settings.set_gf_params(user_config)

In [ ]:
from Garfield.model import Garfield

# Initialize model
model = Garfield(dict_config)

In [ ]:
# Train model
model.train()

In [ ]:
# Compute latent neighbor graph
latent_key = 'garfield_latent'
sc.pp.neighbors(model.adata,
                use_rep=latent_key,
                key_added=latent_key)
# Compute UMAP embedding
sc.tl.umap(model.adata,
           neighbors_key=latent_key)

In [ ]:
# Compute latent Leiden clustering
latent_leiden_resolution = 0.5
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"
latent_key = "garfield_latent"
cell_type_key = 'cell_type'

# louvain leiden
sc.tl.leiden(adata=model.adata,
             resolution=latent_leiden_resolution,
             key_added=latent_cluster_key,
             neighbors_key=latent_key)
len(model.adata.obs[latent_cluster_key].unique())

In [ ]:
# Compute latent Leiden clustering
latent_leiden_resolution = 0.5
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"
latent_key = "garfield_latent"
cell_type_key = 'cell_type'

# louvain leiden
sc.tl.leiden(adata=model.adata,
             resolution=latent_leiden_resolution,
             key_added=latent_cluster_key,
             neighbors_key=latent_key)
len(model.adata.obs[latent_cluster_key].unique())

### Visualize Garfield Latent Space

In [ ]:
import re

from matplotlib import pyplot as plt
import matplotlib
matplotlib.use("Agg") #使用非交互式的后端生成图像文件

def pic(pdf):
    searchObj = re.search( r'(.*).pdf', pdf)
    png = f"{searchObj.group(1)}.png"
    plt.savefig(pdf, bbox_inches="tight")
    plt.savefig(png, bbox_inches="tight", dpi=300)
    plt.close()

In [ ]:
model.adata.obs['niche'].value_counts()

In [ ]:
model.adata

In [ ]:
sc.settings.set_figure_params(dpi=100, facecolor='white')

sc.pl.umap(model.adata, color=['batch', 'cell_type', 'niche', latent_cluster_key],
           s=10, show=False, ncols=2, wspace=0.5) # , legend_loc='on data'

In [ ]:
sc.pl.umap(model.adata, color=[ 'latent_leiden_0.5'],
           s=10, show=False, ncols=2, wspace=0.5, legend_loc='on data') #

In [ ]:
sc.settings.set_figure_params(dpi=100, facecolor='white')

sc.pl.umap(model.adata, color=['cell_type_original'],
           s=10, show=False, ncols=2, wspace=0.5) # , legend_loc='on data'
pic(os.path.join(workdir, "01.umap_plot_ori_celltype.pdf"))

In [ ]:
model.adata.obs['batch'].value_counts()

In [ ]:
model.adata

In [ ]:
cell_type_colors = create_new_color_dict(
    adata=model.adata,
    color_palette="cell_type_20",
    cat_key='niche')

In [ ]:
### 堆积barplot
tmp = pd.crosstab(model.adata.obs["latent_leiden_0.5"],
                  model.adata.obs['niche'], normalize='index')
# tmp = tmp.reindex(model.adata.uns["dendrogram_niche"]["categories_ordered"][::])
ax = tmp.plot.barh(color=cell_type_colors, stacked=True, figsize=(6, 10)).legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.yticks(fontsize=20)
plt.ylabel("Niches", fontsize=20)
plt.xlabel("Original niche Proportions", fontsize=20)
# plt.savefig(f"{workdir}/niche_cell_type_proportions.svg", bbox_inches='tight')

In [ ]:
cell_type_colors = {'endothelial': '#FEE2DDFF',
                    'myeloid': '#EB5291FF',
                    'plasmablast': '#FBBB68FF',
                    'neutrophil': '#C3EF00FF',
                    'NK/T cell': '#9DDAF5FF',
                    'fibroblast': '#6351A0FF',
                    'epithelial': '#FEF79EFF',
                    'B-cell': '#972C8DFF',
                    'mast': '#026CCBFF',
                    'tumor': '#C40003FF',
                    '-1': '#E1D9D1'}

In [ ]:
### 堆积barplot
tmp = pd.crosstab(model.adata.obs["latent_leiden_0.5"],
                  model.adata.obs['cell_type'], normalize='index')
# tmp = tmp.reindex(model.adata.uns["dendrogram_niche"]["categories_ordered"][::])
ax = tmp.plot.barh(color=cell_type_colors, stacked=True, figsize=(6, 10)).legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.yticks(fontsize=20)
plt.ylabel("Niches", fontsize=20)
plt.xlabel("Cell Type Proportions", fontsize=20)
plt.savefig(f"{workdir}/niche_cell_type_proportions.svg", bbox_inches='tight')

In [ ]:
condition_key = 'batch'
condition_colors = {'lung5_rep1': '#7FD2FFFF',
                    'lung5_rep2': '#EAC862FF',
                    'lung5_rep3': '#BA6222FF',
                    'lung6': '#ffd1d7',
                    'lung9_rep1': '#b8396b',
                    'lung9_rep2': '#894FC6FF',
                    'lung12': '#B2DF8AFF'}
tmp = pd.crosstab(model.adata.obs["latent_leiden_0.5"],
                  model.adata.obs[condition_key], normalize='index')
# tmp = tmp.reindex(model.adata.uns["dendrogram_niche"]["categories_ordered"][::])
ax = tmp.plot.barh(color=condition_colors, stacked=True, figsize=(6, 10)).legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.yticks(fontsize=20)
# plt.xticks(fontsize=18)
plt.ylabel("Niches", fontsize=20)
plt.xlabel("Batch Proportions", fontsize=20)
plt.savefig(f"{workdir}/niche_condition_proportions.svg", bbox_inches='tight')

In [ ]:
niches_colors = create_new_color_dict(
    adata=model.adata,
    color_palette="cell_type_20",
    cat_key='latent_leiden_0.5')

In [ ]:
list(model.adata.obs['batch'].unique())

#### celltype 的空间分布图

In [ ]:
cell_type_colors = {'endothelial': '#FEE2DDFF',
                    'myeloid': '#EB5291FF',
                    'plasmablast': '#FBBB68FF',
                    'neutrophil': '#C3EF00FF',
                    'NK/T cell': '#9DDAF5FF',
                    'fibroblast': '#6351A0FF',
                    'epithelial': '#FEF79EFF',
                    'B-cell': '#972C8DFF',
                    'mast': '#026CCBFF',
                    'tumor': '#C40003FF',
                    '-1': '#E1D9D1'}

In [ ]:
import matplotlib.pyplot as plt

# 提取 batch 数据，避免重复筛选
cluster_list = list(model.adata.obs['batch'].unique())  # ['0', '1', '3']

# 创建绘图对象，优化 figsize 以适应子图数量
fig, ax_list = plt.subplots(
    1, len(cluster_list), figsize=(4 * len(cluster_list), 4)
)
ax_list = ax_list.flatten()  # 展平以便在循环中逐一使用

# 定义一个绘图函数，减少重复代码
def plot_embedding(tmp, ax, title, color_key, palette):
    sc.pl.embedding(
        tmp,
        basis='spatial',
        color=color_key,
        palette=palette,
        ax=ax,
        show=False,  # 不立即显示
        s=20,
        title=title,
        legend_loc=None
    )

# 遍历并绘制每个 batch 的图像
for cluster, ax in zip(cluster_list, ax_list):
    tmp = model.adata[model.adata.obs['batch'].isin([cluster]),]

    # 绘制 cell_type 图像
    plot_embedding(tmp, ax, title=f"Donor {cluster}", color_key='cell_type', palette=cell_type_colors)

    # 绘制 latent_leiden_0.5 图像
    # plot_embedding(tmp, ax, title=f"Donor {cluster}", color_key='latent_leiden_0.5', palette=niches_colors)

# 调整布局
plt.tight_layout(w_pad=0.3)
plt.show()  # 统一显示所有图像
pic(os.path.join(workdir, "01.spatial_plot_celltype_each.pdf"))

#### Niches 的空间分布图

In [ ]:
import matplotlib.pyplot as plt

# 提取 batch 数据，避免重复筛选
cluster_list = list(model.adata.obs['batch'].unique())  # ['0', '1', '3']

# 创建绘图对象，优化 figsize 以适应子图数量
fig, ax_list = plt.subplots(
    1, len(cluster_list), figsize=(4 * len(cluster_list), 4)
)
ax_list = ax_list.flatten()  # 展平以便在循环中逐一使用

# 定义一个绘图函数，减少重复代码
def plot_embedding(tmp, ax, title, color_key, palette):
    sc.pl.embedding(
        tmp,
        basis='spatial',
        color=color_key,
        palette=palette,
        ax=ax,
        show=False,  # 不立即显示
        s=20,
        title=title,
        legend_loc=None
    )

# 遍历并绘制每个 batch 的图像
for cluster, ax in zip(cluster_list, ax_list):
    tmp = model.adata[model.adata.obs['batch'].isin([cluster]),]

    # 绘制 cell_type 图像
    # plot_embedding(tmp, ax, title=f"Donor {cluster}", color_key='cell_type', palette=cell_type_colors)

    # 绘制 latent_leiden_0.5 图像
    plot_embedding(tmp, ax, title=f"Donor {cluster}",
                   color_key='latent_leiden_0.5',
                   palette=niches_colors)

# 调整布局
plt.tight_layout(w_pad=0.3)
plt.show()  # 统一显示所有图像
pic(os.path.join(workdir, "01.spatial_plot_niches_each.pdf"))

In [ ]:
sc.pl.umap(model.adata, color=['batch', 'cell_type', 'niche', latent_cluster_key],
           s=10, show=False, ncols=2, wspace=0.5)

In [ ]:
## 注释 niches
## niche 注释
latent_leiden_resolution = 0.5
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"

cluster2annotation = {
    '0': 'Lung9_tumor', #
    '1': 'Lung6_tumor', #
    '2': 'Neu_fibro_mixing', #
    '3': 'Imm_cell_enriched', #
    '4': 'Fibro_plasma_mixing', #
    '5': 'Lung5_tumor', #
    '6': 'Neutrophil_expansion', #
    '7': 'Lymphoid_aggregates', #
    '8': 'Myeloid_enriched', #
    '9': 'Memory_CD8T_tumor', #
    '10': 'Fibro_Treg_mixing', #
    '11': 'Neu_infiltrated_tumor', #
    '12': 'Endo_enriched_tumor',
    '13': 'Lung12_tumor', #
    '14': 'Epithelial_enriched', #
    '15': 'Mast_enriched'
}
model.adata.obs['niche_type'] = model.adata.obs[latent_cluster_key].map(cluster2annotation).astype('category')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sc.tl.dendrogram(adata=model.adata,
                 use_rep="garfield_latent",
                 linkage_method="ward",
                 groupby="latent_leiden_0.5")

fig, (ax) = plt.subplots(1, 1, figsize=(3, 8))
sc.pl.dendrogram(
    adata=model.adata,
    groupby="latent_leiden_0.5",
    orientation="left",
    ax=ax,
    # save="dendrogram_nichecompass_latent.svg"
)
plt.show()

#### 找 niche 差异基因

In [ ]:
# import scanpy as sc
sc.pp.normalize_total(adata_new, target_sum=1e4)
sc.pp.log1p(adata_new)

In [ ]:
latent_leiden_resolution = 0.5
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"

adata_new.obs[latent_cluster_key] = model.adata.obs[latent_cluster_key]
sc.tl.rank_genes_groups(adata_new, latent_cluster_key, method='wilcoxon')

In [ ]:
df  = sc.get.rank_genes_groups_df(adata_new, group=None)
df = df.sort_values(by="scores", ascending=False)
df.to_csv(os.path.join(workdir, 'diff_niches_reference.csv'), index=0, sep='\t')

In [ ]:
def extract_top_genes_spatial(
    adata,
    workdir,
    latent_cluster_key,
    n_top_markers=5,
    output_file="all_marker_genes_spatial_data.csv",
    use_normalized=False
):
    """
    提取 AnnData 对象中所有 marker 基因的表达数据、空间位置信息和 cluster 信息，并保存为 CSV 文件。

    Parameters:
    -----------
    adata : AnnData
        包含 rank_genes_groups 和空间信息的 AnnData 对象。
    workdir : str
        输出文件的保存目录。
    latent_cluster_key : str
        cluster 信息所在的 obs 键。
    output_file : str
        保存的 CSV 文件名。
    use_normalized : bool
        如果为 True，使用标准化后的数据 (.X)；否则使用 counts 层数据。

    Returns:
    --------
    None
    """
    import os
    import pandas as pd

    # 检查 rank_genes_groups 是否存在
    if "rank_genes_groups" not in adata.uns:
        raise ValueError("The AnnData object does not contain 'rank_genes_groups' in .uns.")

    # 提取所有 marker 基因
    ranked_genes = adata.uns['rank_genes_groups']
    marker_genes = set()  # 使用集合避免重复
    clusters = ranked_genes['names'].dtype.names  # 获取所有 cluster 名
    for cluster in clusters:
        top_genes = ranked_genes['names'][cluster][:n_top_markers]  # 获取Top n 基因名
        print(f'Top markers of cluster {cluster}: {top_genes}')
        marker_genes.update(top_genes)  # 添加当前 cluster 的 marker 基因

    marker_genes = list(marker_genes)  # 转换为列表

    # 提取表达数据
    if use_normalized:
        gene_expression = adata[:, marker_genes].X  # 标准化数据
    else:
        gene_expression = adata[:, marker_genes].layers['counts']  # counts 数据

    # 提取空间位置信息
    spatial_positions = adata.obsm['spatial']

    # 提取 cluster 信息
    clusters = adata.obs[latent_cluster_key]

    # 构建最终 DataFrame
    final_dataframe = pd.DataFrame(
        gene_expression.toarray() if hasattr(gene_expression, "toarray") else gene_expression,
        columns=marker_genes,
        index=adata.obs.index
    )
    final_dataframe['x'] = spatial_positions[:, 0]  # 添加 x 坐标
    final_dataframe['y'] = spatial_positions[:, 1]  # 添加 y 坐标
    final_dataframe['cluster'] = clusters.values  # 添加 cluster 信息

    # 保存为 CSV 文件
    os.makedirs(workdir, exist_ok=True)  # 确保工作目录存在
    final_dataframe.to_csv(os.path.join(workdir, output_file))
    print(f"Data saved to {os.path.join(workdir, output_file)}")

    return final_dataframe

In [ ]:
## output res
final_dataframe = extract_top_genes_spatial(
    adata_new,
    workdir,
    latent_cluster_key,
    n_top_markers=5,
    output_file="top_genes_spatial_data_reference.csv",
    use_normalized=False
)

In [ ]:
adata_new.obs['batch'].unique()

In [ ]:
marker = {'0': ['MIF', 'RPL22', 'SOX4', 'HSP90AB1', 'IFI27'],
          '1': ['EGFR' ,'GPNMB', 'KRT6A', 'KRT17' ,'KRT5'],
          '2': ['DUSP5', 'MZT2A', 'WIF1' ,'FKBP11', 'RAMP1'],
          '3': ['MALAT1', 'TYK2', 'B2M', 'CXCR4' ,'CD74'],
          '4': ['COL3A1', 'COL1A1' ,'BGN' ,'TIMP1' ,'IGFBP7'],
          '5': ['S100A6', 'CEACAM6' ,'KRT19', 'OLFM4' ,'MMP7'],
          '6': ['CXCL8' ,'COL9A2', 'SRGN' ,'MZT2A' ,'DUSP5'],
          '7': ['CD74', 'B2M' ,'HLA-DRB1' ,'HLA-A' ,'VIM'],
          '8': ['CD74' ,'HLA-DPA1' ,'HLA-DRB1' ,'HLA-DRA' ,'HLA-DPB1'],
          '9': ['RPL22', 'MIF' ,'NDRG1' ,'SLPI', 'SOX4'],
          '10': ['COL1A1' ,'COL3A1' ,'COL1A2', 'COL6A2', 'COL6A3'],
          '11': ['CXCL8' ,'IGHG1' ,'IL1RN' ,'HCAR2', 'CCL3L3'],
          '12': ['IGFBP7', 'VIM' ,'MGP' ,'SPARCL1' ,'PECAM1'],
          '13': ['S100A6', 'SERPINA1', 'KRT7', 'DUSP5', 'WIF1'],
          '14': ['LTF' ,'SERPINA1' ,'ITGB6' ,'CD74' ,'MMP7'],
          '15': ['TPSB2' ,'TPSAB1', 'CPA3', 'KIT', 'IL1RL1']
          }

In [ ]:
?sc.pl.dotplot

In [ ]:
sc.settings.set_figure_params(dpi=100, facecolor='white')

sc.pl.dotplot(adata_new, marker,
              groupby='latent_leiden_0.5',
              dendrogram=False, cmap="Reds",
              standard_scale='var')
pic(os.path.join(workdir, "01.niche_marker_dotplot.pdf"))

In [ ]:
## niche 1 和 13 是lung6 和 lung12 特异的
gene_list = ["EGFR", "GPNMB"]
fig, ax_list = plt.subplots(
    1, len(gene_list), figsize=(4 * len(gene_list), 4)
)
ax_list = ax_list.flatten()  # 展平以便在循环中逐一使用

tmp = adata_new[adata_new.obs['batch'].isin(['lung6']), ].copy()
for gene, ax in zip(gene_list, ax_list):
    sc.pl.embedding(tmp, basis='spatial', color=gene,
                    ax=ax, show=False, color_map='YlOrRd', s=20)

plt.tight_layout(w_pad=0.3)
# plt.show()
# pic(os.path.join(workdir, "04.spatial_plot_each_niche_marker.pdf"))

In [ ]:
## niche 1 和 13 是lung6 和 lung12 特异的
gene_list = ["S100A6", "SERPINA1"]
fig, ax_list = plt.subplots(
    1, len(gene_list), figsize=(4 * len(gene_list), 4)
)
ax_list = ax_list.flatten()  # 展平以便在循环中逐一使用

tmp = adata_new[adata_new.obs['batch'].isin(['lung12']), ].copy()
for gene, ax in zip(gene_list, ax_list):
    sc.pl.embedding(tmp, basis='spatial', color=gene,
                    ax=ax, show=False, color_map='YlOrRd', s=20)

plt.tight_layout(w_pad=0.3)
# plt.show()
# pic(os.path.join(workdir, "04.spatial_plot_each_niche_marker.pdf"))

In [ ]:
model.adata

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sankey(
    x=model.adata.obs["cell_type"],
    y=model.adata.obs["latent_leiden_0.5"],
    title="",
    title_left="Original\nCellType",
    title_right="Garfield\nNiches",
    ax=ax,
    fontsize="16",  # "xx-small",
    #left_order=model.adata.obs[cell_type_key].unique().tolist(),
    colors=cell_type_colors,
    alpha=0.5)
plt.tight_layout()
pic(os.path.join(workdir, "03.sankey_plot_celltype_niches.pdf"))

### 空转数据之间 niches transfer

In [ ]:
from Garfield.model import Garfield

workdir = f'/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_NSCLC'
gf.settings.set_workdir(workdir)
model_folder_path = f"{workdir}/model"

model = Garfield.load(dir_path=model_folder_path,
              adata_file_name="adata_nsclc.h5ad")

In [ ]:
model.adata

In [ ]:
# read spatial data
import scipy.sparse as sp
### spatial query mapping 选择lung13 样本作为mapping
query_adata = adata[adata.obs['batch'].isin(['lung13'])].copy()
query_adata.var_names_make_unique()
query_adata

In [ ]:
# Ensure adata.X is counts.
# query_adata.layers['counts'] = query_adata.X.copy()
query_adata.X = query_adata.layers['counts'].copy()
query_adata.X.max()

#### Integrating spatially resolved transcriptomics data using Garfield

In [ ]:
new_model = model.load_query_data(dir_path=model_folder_path,
                                  query_adata=query_adata,
                                  ref_adata_name="adata_nsclc.h5ad",
                                  use_cuda=True,
                                  unfreeze_all_weights=False,
                                  unfreeze_eps_weight=True,
                                  unfreeze_layer0=True,
                                  used_mmd=True,
                                  sample_col='projection',
                                  lambda_omics_recon_mmd_loss=1.0)
# Training and obtain latent representation
new_model.train()

# plot UMAP
sc.pp.neighbors(new_model.adata, use_rep='garfield_latent')
sc.tl.umap(new_model.adata)
sc.pl.umap(new_model.adata,
           color=['projection', 'latent_leiden_0.5','niche_type'],
           ncols=1, wspace=0.20, edges=False)

In [ ]:
sc.pl.umap(new_model.adata,
           color=['projection', 'latent_leiden_0.5','niche_type'],
           ncols=1, wspace=0.20, edges=False)

In [ ]:
## split
adata_ref = new_model.adata[new_model.adata.obs['projection'] == 'reference', :]
adata_query = new_model.adata[new_model.adata.obs['projection'] == 'query', :]
spatial_tmp = query_adata.obsm['spatial'].copy()

In [ ]:
### Label transfer
## major celltype
adata_query = new_model.label_transfer(ref_adata=adata_ref,
                                       ref_adata_emb='garfield_latent',
                                       query_adata=adata_query,
                                       query_adata_emb='garfield_latent',
                                       n_neighbors=10,
                                       ref_adata_obs=adata_ref.obs,
                                       label_keys='niche_type')

In [ ]:
adata_query

In [ ]:
adata_query.obsm['spatial'] = spatial_tmp
import matplotlib.pyplot as plt
# adata_query.obsm['spatial'][:, 1] *= -1  # 翻转 y 坐标 手动修正

sc.settings.set_figure_params(dpi=100, facecolor='white')
sc.pl.embedding(adata_query, basis="spatial",
                color=["transferred_niche_type_unfiltered"],
                ncols=1, wspace=0.20, edges=False)

sc.pl.umap(adata_query, color=["transferred_niche_type_unfiltered"],
           ncols=1, wspace=0.20, edges=False)

In [ ]:
adata_ref

In [ ]:
### Label transfer
##  celltype
adata_query = new_model.label_transfer(ref_adata=adata_ref,
                                       ref_adata_emb='garfield_latent',
                                       query_adata=adata_query,
                                       query_adata_emb='garfield_latent',
                                       n_neighbors=10,
                                       ref_adata_obs=adata_ref.obs,
                                       label_keys='cell_type')

In [ ]:
adata_query

In [ ]:
# adata_query.obsm['spatial'] = spatial_tmp
import matplotlib.pyplot as plt
# adata_query.obsm['spatial'][:, 1] *= -1  # 翻转 y 坐标 手动修正

sc.settings.set_figure_params(dpi=100, facecolor='white')
sc.pl.embedding(adata_query, basis="spatial",
                color=["transferred_cell_type_unfiltered"],
                ncols=1, wspace=0.20, edges=False)
sc.pl.umap(adata_query, color=["cell_type"],
           ncols=1, wspace=0.20, edges=False)
sc.pl.umap(adata_query, color=["transferred_cell_type_unfiltered"],
           ncols=1, wspace=0.20, edges=False)

In [ ]:
cell_type_colors = {'endothelial': '#FEE2DDFF',
                    'myeloid': '#EB5291FF',
                    'plasmablast': '#FBBB68FF',
                    'neutrophil': '#C3EF00FF',
                    'NK/T cell': '#9DDAF5FF',
                    'fibroblast': '#6351A0FF',
                    'epithelial': '#FEF79EFF',
                    'B-cell': '#972C8DFF',
                    'mast': '#026CCBFF',
                    'tumor': '#C40003FF',
                    '-1': '#E1D9D1'}

condition_colors = {'lung5_rep1': '#7FD2FFFF',
                    'lung5_rep2': '#EAC862FF',
                    'lung5_rep3': '#BA6222FF',
                    'lung6': '#ffd1d7',
                    'lung9_rep1': '#b8396b',
                    'lung9_rep2': '#894FC6FF',
                    'lung12': '#B2DF8AFF'}

cell_type_original_color = {
    'tumor 5':'#db4c4c',
     'tumor 6':'#e06666',
     'tumor 9':'#e57f7f',
     'tumor 12':'#ea9999',
     'tumor 13':'#efb2b2',
     'fibroblast':'#fff5c4',
     'macrophage':'#53c5ce',
     'T CD4 memory':'#cdd4f6',
     'T CD8 memory':'#ff79a6',
     'plasmablast':'#1a70c2',
     'B-cell':'#f9d6e0',
     'mast':'#b5e3e6',
     'Treg':'#fff49f',
     'endothelial':'#beffa5',
     'pDC':'#d89eff',
     'T CD4 naive':'#fdc0ff',
     'neutrophil':"#DF84A8FF",
     'T CD8 naive':"#F9D078FF",
     'NK':"#EEB0A1FF",
     'monocyte':"#E3EDF6FF",
     'epithelial':"#6351A0FF",
     'mDC':"#972C8DFF"
}

In [ ]:
tmp = adata_query[adata_query.obs["cell_type"].isin(['tumor']), :].copy()
tmp = tmp[tmp.obs["transferred_niche_type_unfiltered"].isin(['Lung9_tumor',
                                                             'Memory_CD8T_tumor',
                                                             'Lung5_tumor',
                                                             'Epithelial_enriched',
                                                             'Lung6_tumor',
                                                             'Lung12_tumor']), :].copy()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sankey(
    x=tmp.obs["cell_type"],
    y=tmp.obs["transferred_niche_type_unfiltered"],
    title="",
    title_left="Original\nMajorCellType",
    title_right="Transfer\nNiches",
    ax=ax,
    fontsize="16",  # "xx-small",
    #left_order=model.adata.obs[cell_type_key].unique().tolist(),
    colors=cell_type_colors,
    alpha=0.5)
plt.tight_layout()
pic(os.path.join(workdir, "03.sankey_plot_celltype_niches.pdf"))

In [ ]:
sc.settings.set_figure_params(dpi=100, facecolor='white')

fig, ax = plt.subplots(figsize=(10, 6))

sankey(
    x=tmp.obs["cell_type"],
    y=tmp.obs["transferred_niche_type_unfiltered"],
    title="",
    title_left="Original\nMajorCellType",
    title_right="Transfer\nNiches",
    ax=ax,
    fontsize="16",  # "xx-small",
    #left_order=model.adata.obs[cell_type_key].unique().tolist(),
    colors=cell_type_colors,
    alpha=0.5)
plt.tight_layout()
pic(os.path.join(workdir, "03.sankey_plot_tumor_niches.pdf"))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sankey(
    x=adata_query.obs["cell_type"],
    y=adata_query.obs["transferred_cell_type_unfiltered"],
    title="",
    title_left="Original\nMajorCellType",
    title_right="Transfer\nMajorCellType",
    ax=ax,
    fontsize="16",  # "xx-small",
    #left_order=model.adata.obs[cell_type_key].unique().tolist(),
    colors=cell_type_colors,
    alpha=0.5)
plt.tight_layout()
# pic(os.path.join(workdir, "03.sankey_plot_celltype_niches.pdf"))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sankey(
    x=adata_query.obs["cell_type_original"],
    y=adata_query.obs["transferred_cell_type_original_unfiltered"],
    title="",
    title_left="Original\nMinCellType",
    title_right="Transfer\nMinCellType",
    ax=ax,
    fontsize="16",  # "xx-small",
    #left_order=model.adata.obs[cell_type_key].unique().tolist(),
    colors=cell_type_original_color,
    alpha=0.5)
plt.tight_layout()
# pic(os.path.join(workdir, "03.sankey_plot_celltype_niches.pdf"))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sankey(
    x=adata_query.obs["cell_type"],
    y=adata_query.obs["transferred_niche_type_unfiltered"],
    title="",
    title_left="Original\nMajorCellType",
    title_right="Transfer\nNiches",
    ax=ax,
    fontsize="16",  # "xx-small",
    #left_order=model.adata.obs[cell_type_key].unique().tolist(),
    colors=cell_type_colors,
    alpha=0.5)
plt.tight_layout()
pic(os.path.join(workdir, "03.sankey_plot_transfer_Majcelltype_niches.pdf"))

In [ ]:
sc.pl.umap(adata_query, color=["transferred_cell_type_original_unfiltered"],
           palette=cell_type_original_color,
           ncols=1, wspace=0.20, edges=False)

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12,6))
ax = sc.pl.umap(adata_query, color=['transferred_cell_type_unfiltered'], size=1.5, frameon=False, ncols=1, wspace=3, ax=axes[0,0], show=False)
ax = sc.pl.umap(adata_query, color=['transferred_cell_type_uncert'], size=1.5, frameon=False, ncols=1, wspace=3, vmax=1, vmin=0, ax=axes[0,1], show=False)
ax = sc.pl.umap(adata_query, color=['transferred_cell_type_original_unfiltered'], size=1.5, frameon=False, ncols=1, wspace=3, ax=axes[1,0], show=False)
ax = sc.pl.umap(adata_query, color=['transferred_cell_type_original_uncert'], size=1.5, frameon=False, ncols=1, wspace=3, vmax=1, vmin=0, ax=axes[1,1], show=False)

fig.tight_layout()

In [ ]:
# Save trained model
model_folder_path = f"{workdir}/model_transfer"
os.makedirs(model_folder_path, exist_ok=True)

new_model.save(dir_path=model_folder_path,
           overwrite=True,
           save_adata=True,
           adata_file_name="adata_concat.h5ad")

In [ ]:
from Garfield.model import Garfield

workdir = f'/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_NSCLC'
gf.settings.set_workdir(workdir)
model_folder_path = f"{workdir}/model_transfer"

new_model = Garfield.load(dir_path=model_folder_path,
                      adata_file_name="adata_concat.h5ad")

#### 输出结果供 R绘图

In [ ]:
### 输出 projection 和 空间位置数据供R绘图
# Extracting 'latent_leiden_0.3' and 'spatial' data
latent_leiden_data = new_model.adata.obs['projection']
spatial_data = new_model.adata.obsm["X_umap"]

# Creating a DataFrame for R visualization
output_df = pd.DataFrame({
    'projection': latent_leiden_data,
    "umap_1": spatial_data[:, 0],
    "umap_2": spatial_data[:, 1],
})

# Save to CSV
output_file = f"latent_projection_umap_data.csv"
output_df.to_csv(os.path.join(workdir, output_file))

In [ ]:
cell_type_key = 'cell_type'
domain_key = 'latent_leiden_0.5'
dataset_name = 'NSCLC'
condition_key = 'batch'

In [ ]:
### 输出celltype 和 空间位置数据供R绘图
# Extracting 'latent_leiden_0.3' and 'spatial' data
latent_leiden_data = adata_query.obs[cell_type_key]
niche_data = adata_query.obs[domain_key]
spatial_data = adata_query.obsm["spatial"]

# Creating a DataFrame for R visualization
output_df = pd.DataFrame({
    domain_key: niche_data,
    cell_type_key: latent_leiden_data,
    "spatial_X": spatial_data[:, 0],
    "spatial_Y": spatial_data[:, 1],
})

# Save to CSV
output_file = f"latent_niche_celltype_data_{dataset_name}.csv"
output_df.to_csv(os.path.join(workdir, output_file))

In [ ]:
adata_query

In [ ]:
adata_query.obs['transferred_cell_type_original_unfiltered'].value_counts()

In [ ]:
### 输出celltype 和 UMAP 位置数据供R绘图
latent_leiden_data = adata_query.obs['transferred_cell_type_unfiltered']
ori_majorcelltype_data = adata_query.obs['cell_type']
ori_mincelltype_data = adata_query.obs['cell_type_original']
niche_data = adata_query.obs['transferred_niche_type_unfiltered']
niche_data_uncert = adata_query.obs['transferred_niche_type_uncert']
spatial_data = adata_query.obsm["X_umap"]

# Creating a DataFrame for R visualization
output_df = pd.DataFrame({
    'transfer_niche': niche_data,
    'transfer_niche_uncert':niche_data_uncert,
    'ori_major_celltype':ori_majorcelltype_data,
    'ori_min_celltype':ori_mincelltype_data,
    'transfer_celltype': latent_leiden_data,
    "umap_1": spatial_data[:, 0],
    "umap_2": spatial_data[:, 1],
})

# Save to CSV
output_file = f"latent_niche_transfer_celltype_umap_data.csv"
output_df.to_csv(os.path.join(workdir, output_file))

In [ ]:
### 输出metadata
metadata=adata_query.obs.copy()

# Save to CSV
output_file = "transfer_metadata_NSCLC.csv"
metadata.to_csv(os.path.join(workdir, output_file))

### 做tensor 分解分析

In [ ]:
from tensorly.decomposition import non_negative_tucker
import tensorly as tl
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import itertools
# %pylab inline

In [ ]:
model.adata

In [ ]:
model.adata.obs['patient'].value_counts()

In [ ]:
### 去除 Niche1 和 Niche13，因为它几乎只在一个样本里富集
cells = model.adata.obs.loc[:, ['batch', 'cell_type_original', 'latent_leiden_0.5','niche_type']]
# 确保 'latent_leiden_0.5' 和 'niche_type' 都是字符串类型
cells['latent_leiden_0.5'] = cells['latent_leiden_0.5'].astype(str)
cells['niche_type'] = cells['niche_type'].astype(str)

# 然后进行拼接
cells['Niches'] = cells['latent_leiden_0.5'] + '-' + cells['niche_type']
cells


In [ ]:
cells['batch'] = cells['batch'].astype('category')
cells['Niches'] = cells['Niches'].astype('category')
cells['cell_type_original'] = cells['cell_type_original'].astype('category')

In [ ]:
list(cells['Niches'].unique())

In [ ]:
# select the cts
cts = list(cells['cell_type_original'].unique())

# select the cns
cns = ['7-Lymphoid_aggregates',
 '3-Imm_cell_enriched',
 '6-Neutrophil_expansion',
 '8-Myeloid_enriched',
 '4-Fibro_plasma_mixing',
 '12-Endo_enriched_tumor',
 '2-Neu_fibro_mixing',
 '14-Epithelial_enriched',
 '11-Neu_infiltrated_tumor',
 '15-Mast_enriched',
 '5-Lung5_tumor',
 '10-Fibro_Treg_mixing',
 '9-Memory_CD8T_tumor',
 '0-Lung9_tumor']

#### Build the tensors for each patient group

In [ ]:
counts = cells.groupby(['batch','Niches','cell_type_original']).size()

In [ ]:
#initialize the tensors
T1 = np.zeros((len(cells['batch'].unique()),len(cns),len(cts)))

for i,pat in enumerate(cells['batch'].unique()):
    for j,cn in enumerate(cns):
        for k,ct in enumerate(cts):
            T1[i,j,k] = counts.loc[(pat,cn,ct)]

#normalize so we have joint distributions each slice
dat1 =np.nan_to_num(T1/T1.sum((1,2), keepdims = True))

In [ ]:
dat1.shape

#### The following tries different numbers of CN modules/CT modules to determine suitable rank for decomposition

In [ ]:
def evaluate_ranks(dat, num_tissue_modules = 2):
    num_tissue_modules = num_tissue_modules+1
    pal = sns.color_palette('bright',10)
    palg = sns.color_palette('Greys',10)

    mat1 = np.zeros((num_tissue_modules,15))
    for i in range(2,15):
        for j in range(1,num_tissue_modules):
            # we use NNTD as described in the paper
            facs_overall = non_negative_tucker(dat,rank=[j,i,i],random_state = 2336)
            mat1[j,i] = np.mean((dat- tl.tucker_to_tensor((facs_overall[0],facs_overall[1])))**2)
    for j in range(1,num_tissue_modules):
        plt.plot(2+np.arange(13),mat1[j][2:],label = 'rank = ({},x,x)'.format(j))

    plt.xlabel('x')
    plt.ylabel('reconstruction error')
    plt.legend()
    plt.show()


In [ ]:
sc.settings.set_figure_params(dpi=100, facecolor='white')

evaluate_ranks(dat1,4)
# plt.show()
# pic(os.path.join(workdir, "00.test_evaluate_ranks.pdf"))

#### The following visualizes the tensor decomposition output
- The exact sizing of the font and aesthetics is difficult to adjust for different sizes and will have to be optimized for number of cell types/cns and intended figure size for your paper.
- Therefore, I have also included a simpler/more conventional visualization of tensor decomposition (plot_modules_heatmap). Only if this analysis is interesting, is it worth proceeding to optimize the layout.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_modules_heatmap(dat, num_tissue_modules=2, num_cn_modules=6):
    # 设置图像大小
    plt.figure(figsize=(20,5))

    # 假设 non_negative_tucker 是已定义的函数，进行矩阵分解
    core, factors = non_negative_tucker(dat, rank=[num_tissue_modules, num_cn_modules, num_cn_modules], random_state=32)

    # 第一个子图：CN模块的加载图
    plt.subplot(1,2,1)
    sns.heatmap(pd.DataFrame(factors[1], index=cns))
    plt.ylabel('CN')
    plt.xlabel('CN module')
    plt.title('Loadings onto CN modules')

    # 第二个子图：CT模块的加载图
    plt.subplot(1,2,2)
    sns.heatmap(pd.DataFrame(factors[2], index=cts))
    plt.ylabel('CT')
    plt.xlabel('CT module')
    plt.title('Loadings onto CT modules')
    plt.show()

    # 设置子图的尺寸
    plt.figure(figsize=(num_tissue_modules*3, 3))

    # 绘制每个组织模块的couplings
    for p in range(num_tissue_modules):
        plt.subplot(1, num_tissue_modules, p+1)
        sns.heatmap(pd.DataFrame(core[p]))
        plt.title(f'tissue module {p}, couplings')
        plt.ylabel('CN module')
        plt.xlabel('CT module')
    plt.show()

In [ ]:
plot_modules_heatmap(dat1, num_tissue_modules=4, num_cn_modules=10)
# pic(os.path.join(workdir, "00.plot_modules_heatmap.pdf"))

In [ ]:
def plot_modules_graphical(dat,num_tissue_modules = 2, num_cn_modules = 6, scale = 0.4):
    core, factors = non_negative_tucker(dat,rank=[num_tissue_modules,num_cn_modules,num_cn_modules],random_state = 32)


    pal = sns.color_palette('bright',20)
    palg = sns.color_palette('Greys',20)

    plt.figure(figsize=(3.67*scale,2.00*scale))
    cn_scatter_size = scale*scale*45
    cel_scatter_size = scale*scale*15

    for p in range(num_tissue_modules):
        for idx in range(num_cn_modules):
            an = float(np.max(core[p][idx,:])>0.1) + (np.max(core[p][idx,:])<=0.1)*0.05
            ac = float(np.max(core[p][:,idx])>0.1) + (np.max(core[p][:,idx])<=0.1)*0.05

            cn_fac = factors[1][:,idx]
            cel_fac = factors[2][:,idx]

            cols_alpha = [(*pal[cn], an*np.minimum(cn_fac, 1.0)[i]) for i,cn in enumerate(cns)]
            cols = [(*pal[cn], np.minimum(cn_fac, 1.0)[i]) for i,cn in enumerate(cns)]
            cell_cols_alpha = [(0,0,0, an*np.minimum(cel_fac, 1.0)[i]) for i,_ in enumerate(cel_fac)]
            cell_cols = [(0,0,0, np.minimum(cel_fac, 1.0)[i]) for i,_ in enumerate(cel_fac)]

            plt.scatter(0.5*np.arange(len(cn_fac)), 5*idx + np.zeros(len(cn_fac)), c = cols_alpha, s = cn_scatter_size)
            offset = 9
            for i,k in enumerate(cns):
                plt.text(0.5*i, 5*idx, k,fontsize = scale*2,ha = 'center', va = 'center',alpha = an)

            plt.scatter(-4.2+0.25*np.arange(len(cel_fac))+offset, 5*idx + np.zeros(len(cel_fac)), c = cell_cols_alpha, s = 0.5*cel_scatter_size)#,vmax = 0.5,edgecolors=len(cell_cols_alpha)*[(0,0,0,min(1.0,max(0.1,2*an)))], linewidths= 0.05)


            rect = plt.Rectangle((-0.5,5*idx-2 ),4.5,4,linewidth=scale*scale*1,edgecolor='black',facecolor='none',zorder = 0,alpha = an,linestyle = '--')
            ax = plt.gca()
            ax.add_artist(rect)
            plt.scatter([offset-5],[5*idx],c = 'black', marker = 'D', s = scale*scale*5, zorder = 5,alpha = an)
            plt.text(offset-5,5*idx,idx,color = 'white',alpha = an, ha = 'center', va = 'center',zorder = 6,fontsize = 4.5)
            plt.scatter([offset-4.5],[5*idx],c = 'black', marker = 'D', s = scale*scale*5, zorder = 5,alpha = ac)
            plt.text(offset-4.5,5*idx,idx,color = 'white',alpha = ac, ha = 'center', va = 'center', zorder = 6,fontsize = 4.5)

            rect = plt.Rectangle((offset-4.5,5*idx-2 ),4.5,4,linewidth=scale*1,edgecolor='black',facecolor='none',zorder = 0, alpha = ac,linestyle = '-.')
            ax.add_artist(rect)

        for i,ct in enumerate(cts):
                plt.text(-4.2+offset+0.25*i, 27.5, ct, rotation = 45, color = 'black',ha = 'left', va = 'bottom',fontsize = scale*2,alpha = 1)
        for cn_i in range(num_cn_modules):
            for cel_i in range(num_cn_modules):
                plt.plot([-3+offset -2, -4+offset - 0.5],[5*cn_i, 5*cel_i], color = 'black', linewidth =2*scale*scale*1* min(1.0, max(0,-0.00+core[p][cn_i,cel_i])),alpha = min(1.0, max(0.000,-0.00+10*core[p][cn_i,cel_i])))#max(an,ac))



        plt.ylim(-5, 30)
        plt.axis('off')


        plt.show()

In [ ]:
cns = [0,2,3,4,5,6,7,8,9,10,11,12,14,15]

In [ ]:
plot_modules_graphical(dat1,scale  = 2)

### 保存model 结果

In [ ]:
# Save trained model
model_folder_path = f"{workdir}/model"
os.makedirs(model_folder_path, exist_ok=True)

model.save(dir_path=model_folder_path,
           overwrite=True,
           save_adata=True,
           adata_file_name="adata_nsclc.h5ad")

In [ ]:
from Garfield.model import Garfield

workdir = f'/pri_exthome/zhouwg/project/Garfield_benchmark/results/sp_unimodal/spRNA_NSCLC'
gf.settings.set_workdir(workdir)
model_folder_path = f"{workdir}/model"

model = Garfield.load(dir_path=model_folder_path,
              adata_file_name="adata_nsclc.h5ad")

### 输出结果供R绘图

In [ ]:
model.adata

In [ ]:
cell_type_key = 'cell_type'
domain_key = 'latent_leiden_0.5'
dataset_name = 'NSCLC'
condition_key = 'batch'

In [ ]:
### 输出celltype 和 空间位置数据供R绘图
# Extracting 'latent_leiden_0.3' and 'spatial' data
latent_leiden_data = model.adata.obs[cell_type_key]
niche_data = model.adata.obs[domain_key]
spatial_data = model.adata.obsm["spatial"]

# Creating a DataFrame for R visualization
output_df = pd.DataFrame({
    domain_key: niche_data,
    cell_type_key: latent_leiden_data,
    "spatial_X": spatial_data[:, 0],
    "spatial_Y": spatial_data[:, 1],
})

# Save to CSV
output_file = f"latent_niche_celltype_data_{dataset_name}.csv"
output_df.to_csv(os.path.join(workdir, output_file))

In [ ]:
### 输出celltype 和 空间位置数据供R绘图
# Extracting 'latent_leiden_0.3' and 'spatial' data
latent_leiden_data = model.adata.obs[cell_type_key]
niche_data = model.adata.obs[domain_key]
condition_data = model.adata.obs[condition_key]
spatial_data = model.adata.obsm["X_umap"]

# Creating a DataFrame for R visualization
output_df = pd.DataFrame({
    domain_key: niche_data,
    cell_type_key: latent_leiden_data,
    condition_key: condition_data,
    "umap_1": spatial_data[:, 0],
    "umap_2": spatial_data[:, 1],
})

# Save to CSV
output_file = f"latent_niche_celltype_umap_data_{dataset_name}.csv"
output_df.to_csv(os.path.join(workdir, output_file))

In [ ]:
def plot_cluster_proportions(cluster_props,
                             cluster_palette=None,
                             xlabel_rotation=0,
                             figsize=(9,4),
                             ax=None,
                             figs=None):
    if ax is None:
        figs, ax = plt.subplots(figsize=figsize)
        figs.patch.set_facecolor("white")
        figs.tight_layout()

    cmap = None
    if cluster_palette is not None:
        cmap = sns.palettes.blend_palette(
            cluster_palette,
            n_colors=len(cluster_palette),
            as_cmap=True)

    cluster_props.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        legend=None,
        colormap=cmap
    )

    ax.legend(bbox_to_anchor=(1.01, 1), frameon=False, title="Cluster").remove()
    ax.tick_params(axis="x", rotation=xlabel_rotation, bottom=False)
    ax.tick_params(axis="y", rotation=90)
    ax.set_xlabel('Niche', fontsize=20)
    ax.set_ylabel("Proportion", fontsize=20)
    ax.tick_params(axis='both', which='major', labelsize=15)
    ax.spines.left.set_bounds(0, 100)
    ax.spines.right.set_visible(False)
    ax.spines.bottom.set_visible(False)
    ax.spines.top.set_visible(False)

    return ax

In [ ]:
cell_type_colors = {'endothelial': '#FEE2DDFF',
                    'myeloid': '#EB5291FF',
                    'plasmablast': '#FBBB68FF',
                    'neutrophil': '#C3EF00FF',
                    'NK/T cell': '#9DDAF5FF',
                    'fibroblast': '#6351A0FF',
                    'epithelial': '#FEF79EFF',
                    'B-cell': '#972C8DFF',
                    'mast': '#026CCBFF',
                    'tumor': '#C40003FF',
                    '-1': '#E1D9D1'}

condition_colors = {'lung5_rep1': '#7FD2FFFF',
                    'lung5_rep2': '#EAC862FF',
                    'lung5_rep3': '#BA6222FF',
                    'lung6': '#ffd1d7',
                    'lung9_rep1': '#b8396b',
                    'lung9_rep2': '#894FC6FF',
                    'lung12': '#B2DF8AFF'}

niches_colors = create_new_color_dict(
    adata=model.adata,
    color_palette="cell_type_20",
    cat_key='latent_leiden_0.5')

In [ ]:
niches_colors

In [ ]:
sc.settings.set_figure_params(dpi=100, facecolor='white')

sc.pl.umap(model.adata, color=['batch'],
           palette=condition_colors,
           s=10, show=False, ncols=2, wspace=0.5)
sc.pl.umap(model.adata, color=['cell_type'],
           palette=cell_type_colors,
           s=10, show=False, ncols=2, wspace=0.5)
sc.pl.umap(model.adata, color=['niche'],
           s=10, show=False, ncols=2, wspace=0.5)
sc.pl.umap(model.adata, color=[domain_key],
           palette=niches_colors,
           s=10, show=False, ncols=2, wspace=0.5)

In [ ]:
tmp = model.adata.copy()

latent_key = 'garfield_latent'
# 计算UMAP（3维）
sc.tl.umap(tmp, n_components=3,
           neighbors_key=latent_key)
# 绘制3D UMAP
# fig = plt.figure(figsize=(10, 8))
# ax = fig.add_subplot(111, projection='3d')
# ax.scatter(tmp.obsm['X_umap'][:, 0], tmp.obsm['X_umap'][:, 1], tmp.obsm['X_umap'][:, 2], c=adata.obs['louvain'].cat.codes, cmap='viridis', s=5)
# ax.set_xlabel('UMAP 1')
# ax.set_ylabel('UMAP 2')
# ax.set_zlabel('UMAP 3')
# plt.show()

In [ ]:
sc.pl.umap(tmp, color=[domain_key],
           palette=niches_colors, projection='3d', # legend_loc='on data',
           s=3, show=False, ncols=2, wspace=0.5)

In [ ]:
model.adata.obs[domain_key].value_counts()

In [ ]:
# Plot proportions for niches 1, 7, and 13
sc.settings.set_figure_params(dpi=100, facecolor='white')

plot_var = f'{domain_key}'

# Iterate over the specified niches
for ids in ['1', '7', '13']:
    tmp = model.adata[model.adata.obs[domain_key].isin([ids]), :].copy()

    # Create a new figure for each niche
    figs, axes = plt.subplots(nrows=1, ncols=2, figsize=(3, 3))

    # Iterate over the cluster variables
    for i, cluster_var in enumerate(['cell_type', 'batch']):
        # Group by cluster and plot variable, then calculate proportions
        props = tmp.obs.groupby([cluster_var, plot_var]).size().reset_index()
        props = props.pivot(columns=plot_var, index=cluster_var).T
        props.index = props.index.droplevel(0)
        props.fillna(0, inplace=True)
        props = props.div(props.sum(axis=1), axis=0) * 100

        # Plot the proportions
        plot_cluster_proportions(
            props, xlabel_rotation=90,
            cluster_palette=tmp.uns[f'{cluster_var}_colors'],
            figsize=(3, 3), ax=axes[i], figs=figs
        )

    # Save the plot for the current niche
    pic(os.path.join(workdir, f"02.stack_plot_prop_celltype_niche_{ids}.pdf"))

    # Close the figure to free up memory for the next iteration
    plt.close(figs)

In [ ]:
# plot proportions
figs, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 3))
plot_var = f'{domain_key}'

for i, cluster_var in enumerate(['cell_type', 'niche', 'batch']):
    props = model.adata.obs.groupby([cluster_var, plot_var]).size().reset_index()
    props = props.pivot(columns=plot_var, index=cluster_var).T
    props.index = props.index.droplevel(0)
    props.fillna(0, inplace=True)
    props = props.div(props.sum(axis=1), axis=0)*100
    axes[i] = plot_cluster_proportions(props, xlabel_rotation=90,
                                       cluster_palette=model.adata.uns[f'{cluster_var}_colors'], figsize=(4,3), ax=axes[i], figs=figs)
figs.show()
pic(os.path.join(workdir, "02.stack_plot_prop_celltype_niche_batch.pdf"))

#### 计算niche中 celltype 的proportion

In [ ]:
import pandas as pd

def calculate_celltype_proportion(confusion_matrix, save_path=None, file_format='csv'):
    """
    计算每个niche中各个celltype的比例，并可选择保存结果。

    参数：
    - confusion_matrix (pd.DataFrame): 行是niches，列是celltypes的混淆矩阵。
    - save_path (str, optional): 保存结果的路径。如果为None，则不保存结果。
    - file_format (str, optional): 保存文件的格式，支持 'csv' 和 'xlsx'，默认为 'csv'。

    返回：
    - proportion_matrix (pd.DataFrame): 每个niche中各个celltype的比例矩阵。
    """
    # 计算每个niche的总数（行的总和）
    niche_totals = confusion_matrix.sum(axis=1)

    # 计算每个celltype在每个niche中的比例
    proportion_matrix = confusion_matrix.div(niche_totals, axis=0)

    # 如果指定了保存路径，保存结果
    if save_path:
        if file_format == 'csv':
            proportion_matrix.to_csv(save_path)
            print(f"结果已保存为 {save_path} (CSV 格式)")
        elif file_format == 'xlsx':
            proportion_matrix.to_excel(save_path)
            print(f"结果已保存为 {save_path} (Excel 格式)")
        else:
            print(f"不支持的文件格式 {file_format}，未保存结果")

    return proportion_matrix

In [ ]:
cell_type_key = 'cell_type_original'
domain_key = 'latent_leiden_0.5'
dataset_name = 'NSCLC'
condition_key = 'batch'

In [ ]:
# 调用函数并保存结果
tmp = model.adata.copy()
df = tmp.obs[[domain_key, cell_type_key]].groupby([domain_key, cell_type_key]).size().unstack(fill_value=0)
proportion_matrix = calculate_celltype_proportion(confusion_matrix=df,
                                                  save_path=f'{workdir}/niche_prop_alldata2.csv',
                                                  file_format='csv')
# 输出 df
df.to_csv(f'{workdir}/niche_alldata2.csv')

# 打印比例矩阵
proportion_matrix

In [ ]:
# 调用函数并保存结果
tmp = model.adata.copy()
df = tmp.obs[[domain_key, cell_type_key]].groupby([domain_key, cell_type_key]).size().unstack(fill_value=0)
proportion_matrix = calculate_celltype_proportion(confusion_matrix=df,
                                                  save_path=f'{workdir}/niche_prop_alldata.csv',
                                                  file_format='csv')
# 输出 df
df.to_csv(f'{workdir}/niche_alldata.csv')

# 打印比例矩阵
proportion_matrix

In [ ]:
### 输出metadata
metadata=model.adata.obs.copy()

# Save to CSV
output_file = "metadata_NSCLC.csv"
metadata.to_csv(os.path.join(workdir, output_file))